In [1]:
! python --version
import scipy

Python 3.10.9
Python 3.10.9


# 0. Set up project and read in data

In [2]:
import pandas as pd
from pathlib import Path
from collections import Counter

# Get the current directory
current_directory = Path.cwd()

# Set directories
dir_sc_dat = current_directory.parent.parent.parent / "6_scRNAseq organoid data"
dir_dat = current_directory.parent.parent.parent / "0_data" 
dir_res = current_directory.parent.parent.parent / "2_results" 

# Read in sc data from Brittney's paper ----
sc_data = pd.read_csv(dir_sc_dat / 'Excel Supplement B genes brittneys paper.csv')

# Get names of genes
genes = sc_data.gene

# Split symbols on '|', and flatten the list
sc_genes = [symbol for item in genes for symbol in item.split('; ')] 

# Get unique Genes
unique_sc_genes = list(Counter(sc_genes).keys())


# Read in eQTL data ----
eqtls = pd.read_csv(dir_dat / "eQTLs" / 'eqtl_rsid_pairs.csv')

# Split symbols on '|', and flatten the list
eqtl_genes = [symbol for item in eqtls.gene for symbol in item.split('; ')] 

# Get unique genes
unique_eqtl_genes = list(Counter(eqtl_genes).keys())


# Combine all genes names ----
combined_genes = unique_sc_genes + unique_eqtl_genes


# Convert to list of strings in the format you want
gene_list = "[" + ", ".join(["'" + symbol + "'" for symbol in combined_genes]) + "]"
gene_list = gene_list.replace(' ', '')

print(gene_list)

['APOA4','PLIN2','FABP1','HMGCS2','APOA2','MT1F','ACSL1','MT1X','APOB','MT1G','CYP3A5','HADHB','MT1H','ACAA2','RPS20','ACAA1','CYP3A4','BAAT','MT1M','FBP1','MT2A','PDK4','SCD','ACADVL','IGFBP1','RPS29','APOC3','PCK1','ALDH2','PRAP1','PPP2R2B','CYBA','SKAP1','INPP4B','PTPRC','CBLB','IKZF1','ATXN1','PRKCB','CD7','CELF2','TOX','GZMA','PLCB1','KCNQ5','PARP8','MBNL1','RUNX1','GSTA1','SFMBT2','ADH1C','BCL2','PIP4K2A','ANKRD44','PITPNC1','AKR1B10','CHST11','ARHGAP15','AOAH','PRKCH','RPL37A','RPL23','RPS27','RPL38','RPL37','RPS21','RPS11','RPS28','RPL39','RPL13A','RPL21','RPS16','RPLP2','RPL31','RPS26','RPL27A','TMA7','SET','RPL36','RPL36A','ATP5ME','HIVEP2','RABGAP1L','FRYL','WWOX','MAML2','JARID2','STK39','MT-ND4L','IKZF2','LRBA','AKR1C1','KLRC2','UTRN','ALDH1A1','PTGR1','PRKCA','HADHA','ANGPTL4','ABCB4','CYP3A7','G6PC','ACADM','CYP4A11','ACOX1','APOA5','SLC25A47','CYP2B6','SERPINE1','AKR1C3','SAA4','AGT','ACTG1','TPM1','TM4SF4','TUBB2A','LINC01435','PKN2-AS1','APOH','CYP2C9','UGT2B10','UGT2

# 1. Query ComptoxAI database using Neo4j.
## 1.a. Identify all nodes linking PFHpA, genes, and the outcome. 
Note: to get this to run, you have to have an Neo4j instance of comptoxai.

In [3]:
import comptox_ai
from comptox_ai.db.graph_db import GraphDB
db = GraphDB(username="cytoscape", password="12345", hostname="localhost:7687")

# Define Disease and Exposure Names for ComptoxAI 
chemical_common_name = 'Perfluoroheptanoic acid'
disease_common_name = 'Liver carcinoma'
max_level = str('3')

# Start and end strings
start_string  = "MATCH (d:Disease {commonName: '" + disease_common_name + "' }) MATCH (end:Gene) WHERE end.geneSymbol IN "
middle_string = " CALL apoc.path.spanningTree(d, {relationshipFilter: '<GENEASSOCIATESWITHDISEASE|GENEINTERACTSWITHGENE', minLevel: 1, maxLevel: " + max_level + ", endNodes: end}) YIELD path WITH nodes(path) as n RETURN n UNION MATCH (c:Chemical {commonName: '" + chemical_common_name + "'}) MATCH (d:Disease {commonName: '" + disease_common_name + "'}) MATCH (end:Gene) WHERE end.geneSymbol IN"
end_string    = " CALL apoc.path.spanningTree(c, {relationshipFilter: 'CHEMICALINCREASESEXPRESSION>|CHEMICALDECREASESEXPRESSION>|GENEINTERACTSWITHGENE>', minLevel: 1, maxLevel: " + max_level + ", endNodes: end}) YIELD path WITH nodes(path) as n RETURN n;"

# Combine the start and end strings with the unique_gene_symbols_list
first_query = start_string + gene_list + middle_string + gene_list + end_string
# print(first_query)

# Run initial query
nodes = db.run_cypher(first_query) 

Attempting to connect to public Neo4j database at `localhost:7687`...
...connection established successfully.
Attempting to connect to public Neo4j database at `localhost:7687`...
...connection established successfully.


## 1.b. Clean the output to get the information on each node individually

In [4]:
# Unlist everything and get a list of separate dictionaries
flattened_list = [inner_dict for outer_dict in nodes for inner_dict in outer_dict['n']]

# Remove duplicates by converting to a set and back to a list
unique_list = list({tuple(item.items()) for item in flattened_list})

# Convert the list of tuples back to dictionaries
unique_list = [dict(item) for item in unique_list]

In [5]:
# Extract geneSymbol
# Create a set to store unique geneSymbol values
unique_gene_symbols = set()

# Iterate through the list and collect unique geneSymbol values
for dictionary in flattened_list:
    if 'geneSymbol' in dictionary:
        unique_gene_symbols.add(dictionary['geneSymbol'])

# Convert the set back to a list if needed
unique_gene_symbols_list = list(unique_gene_symbols)

# Now, 'unique_gene_symbols_list' contains all unique geneSymbol values

print('The number of unique genes is ' +  str(len(unique_gene_symbols_list)) + ", and " + str(len(set(unique_gene_symbols_list) & set(combined_genes))) + " were part of the original query, including:")
print('- ' + str(len(set(unique_gene_symbols_list) & set(unique_sc_genes))) + "/" + str(len(unique_sc_genes)) + " from the single cell data.")
print('- ' + str(len(set(unique_gene_symbols_list) & set(unique_eqtl_genes))) + "/" + str(len(unique_eqtl_genes)) +  " eQTLs.")

The number of unique genes is 223, and 135 were part of the original query, including:
- 125/135 from the single cell data.
- 10/18 eQTLs.
The number of unique genes is 223, and 135 were part of the original query, including:
- 125/135 from the single cell data.
- 10/18 eQTLs.


## 1.c. With all genes, identify all pathways linked to at least some number of genes

In [6]:
start_string = "MATCH (gene:Gene)-[:GENEINPATHWAY]->(pw) WHERE gene.geneSymbol IN"
end_string = " WITH pw, COUNT(DISTINCT gene) AS geneCount WHERE geneCount >= 15 RETURN pw;"

# Combine the start and end strings with the unique_gene_symbols_list
second_query = start_string + "[" + ", ".join(["'" + symbol + "'" for symbol in unique_gene_symbols_list]) + "]" + end_string
# print(second_query)

# Run query
pathway_nodes = db.run_cypher(second_query) 

# Get unique pathway nodes
unique_pathway_ids = list({node['pw']['pathwayId'] for node in pathway_nodes})
print(unique_pathway_ids)

['R-HSA-186797', 'R-HSA-1799339', 'R-HSA-76002', 'path:hsa05205', 'R-HSA-422475', 'R-HSA-72766', 'path:hsa04933', 'R-HSA-927802', 'R-HSA-72737', 'R-HSA-1433557', 'R-HSA-372790', 'R-HSA-556833', 'None', 'R-HSA-194138', 'R-HSA-72689', 'path:hsa03320', 'R-HSA-1280218', 'WP2882', 'R-HSA-2408557', 'R-HSA-975957', 'path:hsa04151', 'path:hsa01100', 'R-HSA-2172127', 'Pathway_EGFR1', 'WP2377', 'R-HSA-157279', 'path:hsa05418', 'R-HSA-2454202', 'R-HSA-1280215', 'R-HSA-2424491', 'R-HSA-72706', 'R-HSA-881907', 'R-HSA-194315', 'R-HSA-975956', 'WP477', 'R-HSA-74160', 'R-HSA-166520', 'R-HSA-1236394', 'R-HSA-192823', 'R-HSA-9010553', 'Leukotriene metabolism', 'path:hsa05200', 'R-HSA-1430728', 'R-HSA-168249', 'R-HSA-156902', 'R-HSA-168256', 'R-HSA-109582', 'path:hsa04510', 'R-HSA-4420097', 'R-HSA-6791226', 'R-HSA-72613', 'R-HSA-1266738', 'R-HSA-72764', 'R-HSA-156827', 'R-HSA-392499', 'R-HSA-162582', 'R-HSA-156842', 'WP2004', 'path:hsa03010']
['R-HSA-186797', 'R-HSA-1799339', 'R-HSA-76002', 'path:hsa0520

## 1.d. With all nodes, identify all of the relationships

In [7]:
# Create cypher query
# Start and end strings
start_string = "MATCH (c:Chemical {commonName: '" + chemical_common_name + "'}) MATCH (d:Disease {commonName: '" + disease_common_name + "'}) MATCH (node1:Gene) WHERE node1.geneSymbol IN "
middle_string = " MATCH (pw1:Pathway) WHERE pw1.pathwayId IN "
end_string_with_pathway = " WITH collect(id(node1))+collect(id(pw1))+collect(c)+collect(d) as nodes CALL apoc.algo.cover(nodes) YIELD rel RETURN  startNode(rel), rel, endNode(rel);"
end_string_no_pathways = " WITH collect(id(node1))+collect(c)+collect(d) as nodes CALL apoc.algo.cover(nodes) YIELD rel RETURN  startNode(rel), rel, endNode(rel);"

# Combine the start and end strings with the unique_gene_symbols_list
query_string_with_pathways = start_string + "[" + ", ".join(["'" + symbol + "'" for symbol in unique_gene_symbols_list]) + "]" + middle_string + "[" + ", ".join(["'" + symbol + "'" for symbol in unique_pathway_ids]) + "]" + end_string_with_pathway
query_string_no_pathways = start_string + "[" + ", ".join(["'" + symbol + "'" for symbol in unique_gene_symbols_list]) + "]" + end_string_no_pathways

# print(query_string_no_pathways)
# print(query_string_with_pathways)

# Run Cypher Query
data = db.run_cypher(query_string_no_pathways)
data_with_pw = db.run_cypher(query_string_with_pathways)

# 2. Create Graph

## 2.a. Create Base Graph from ComptoxAI query

In [65]:
# Create network diagram
import networkx as nx
import matplotlib.pyplot as plt
# Create a new graph
G = nx.DiGraph()

# Function to compute the combined 'type' attribute
def compute_type(node):
    if node.get('commonName') == chemical_common_name:
        return 'PFAS'
    if node.get('commonName') == disease_common_name:
        return 'Disease'
    if node.get('pathwayId') in unique_pathway_ids:
        return 'Pathway'
    if node.get('geneSymbol') in unique_sc_genes:
        return 'Gene: In-vitro'
    # if node.get('geneSymbol') in snp_associated_genes:
    #     return 'Gene: SNP associated gene'
    if node.get('geneSymbol') in unique_eqtl_genes:
        return 'Gene: eQTL'
    else:
        return 'Gene: from ComptoxAI'

    
# Add nodes and edges with combined 'type' attribute
for entry in data:
    start_node = entry['startNode(rel)']
    end_node = entry['endNode(rel)']
    rel_type = entry['rel'][1]  # Relationship type is the second item in the tuple
    
    # Set combined 'type' attribute
    start_node['type'] = compute_type(start_node)
    end_node['type'] = compute_type(end_node)

    # Node identifiers
    start_node_id = start_node.get('geneSymbol')  or start_node.get('commonName')
    end_node_id = end_node.get('geneSymbol') or end_node.get('xrefUmlsCUI') or end_node.get('commonName') 

    # Add nodes with combined 'type' attribute
    G.add_node(start_node_id, **start_node)
    G.add_node(end_node_id, **end_node)

    # # Add edge
    G.add_edge(start_node_id, end_node_id, relationship=rel_type, weight = 1)

## 2.b. Add nodes for SNPS associated with eQTLs

In [66]:
# Filter eqtls to only include genes in the final graph
eqtl_genes_in_graph = set(unique_gene_symbols_list) & set(unique_eqtl_genes)
filtered_eqtls = eqtls[eqtls['gene'].isin(eqtl_genes_in_graph)]

# Assuming 'filtered_eqtls' has columns 'SNP' and 'gene'
for idx, row in filtered_eqtls.iterrows():
    snp = row['rsid']  # The SNP node
    gene = row['gene']  # The gene node (which should already be in the network)
    
    # Check if the gene is already in the graph
    if gene not in G.nodes:
        print(f"Gene {gene} is not in the graph!")
        continue  # Skip this SNP if the gene isn't present

    # print(f"Adding row {idx}: SNP={snp}, gene={gene}")

    # Add the SNP as a new node (you can customize attributes here)
    G.add_node(snp, type='SNP')

    # Add an edge between the SNP and the gene (customize relationship as needed)
    G.add_edge(snp, gene, relationship='SNP-gene association')

## 2.c. Add PFHpA and SNP edges

In [67]:
unique_rsid = list(Counter(filtered_eqtls.rsid).keys())
rsid_pfas_link = pd.DataFrame(unique_rsid, columns= ['SNP'])

# Add edge between exposure and scRNAseq genes
for idx, row in rsid_pfas_link.iterrows():
    snp = row['SNP']  # The SNP node
    exposure = chemical_common_name  # The name of the PFAS node (which is already be in the network)
  
    print(f"Adding edge {idx}: exposure={exposure}, SNP={snp}")

    # Add an edge between the exposure and the SNP 
    G.add_edge(exposure, snp, relationship = 'Exposure-SNP association', weight = 1)



Adding edge 0: exposure=Perfluoroheptanoic acid, SNP=rs4808199
Adding edge 1: exposure=Perfluoroheptanoic acid, SNP=rs56094641
Adding edge 2: exposure=Perfluoroheptanoic acid, SNP=rs738408
Adding edge 0: exposure=Perfluoroheptanoic acid, SNP=rs4808199
Adding edge 1: exposure=Perfluoroheptanoic acid, SNP=rs56094641
Adding edge 2: exposure=Perfluoroheptanoic acid, SNP=rs738408


## 2.d. Add PFHpA and scRNAseq edges

In [68]:
unique_sc_data = list(Counter(sc_data.gene).keys())
unique_sc_data = pd.DataFrame(unique_sc_data, columns= ['gene'])

# Create a new column to store whether the edge exists or not
unique_sc_data['edge_exists'] = False

# Check if edge exists
for idx, row in unique_sc_data.iterrows():
    sc_gene = row['gene']  # The SNP node
    exposure = chemical_common_name  # The name of the PFAS node (which is already be in the network)

    # Check if there is an edge between chemical_common_name and sc_gene
    if G.has_edge(exposure, sc_gene):
        unique_sc_data.at[idx, 'edge_exists'] = True


# Count how many True values there are in the 'edge_exists' column
true_count = unique_sc_data['edge_exists'].sum()

# Create a summary table
summary_table = pd.DataFrame({
    'Total Genes': [len(unique_sc_data)],
    'Edges Found (True)': [true_count],
    'Edges Not Found (False)': [len(unique_sc_data) - true_count]
})

# Display the summary table
print(summary_table)

# Add edge between exposure and scRNAseq genes
# Add or modify edge between exposure and scRNAseq genes
for idx, row in unique_sc_data.iterrows():
    sc_gene = row['gene']  # The gene node
    exposure = chemical_common_name  # The name of the PFAS node

    # Check if the gene is in the graph
    if sc_gene not in G.nodes:
        print(f"Gene {sc_gene} is not in the graph!")
        continue  # Skip this gene if it isn't present

    # Check if there is already an edge between exposure and sc_gene
    if G.has_edge(exposure, sc_gene):
        # If the edge exists, modify the relationship and the weight
        G[exposure][sc_gene]['relationship'] = 'ComptoxAI and Organoid association'
        G[exposure][sc_gene]['weight'] = 2
        print(f"Modified edge {idx}: exposure={exposure}, sc_gene={sc_gene}, relationship='ComptoxAI and Organoid association', weight=2")
    else:
        # If the edge doesn't exist, add it with a different relationship and weight
        G.add_edge(exposure, sc_gene, relationship='Organoid association', weight=1)
        print(f"Added new edge {idx}: exposure={exposure}, sc_gene={sc_gene}, relationship='Organoid association', weight=1")

    

   Total Genes  Edges Found (True)  Edges Not Found (False)
0          135                  14                      121
Added new edge 0: exposure=Perfluoroheptanoic acid, sc_gene=APOA4, relationship='Organoid association', weight=1
Modified edge 1: exposure=Perfluoroheptanoic acid, sc_gene=PLIN2, relationship='ComptoxAI and Organoid association', weight=2
Modified edge 2: exposure=Perfluoroheptanoic acid, sc_gene=FABP1, relationship='ComptoxAI and Organoid association', weight=2
Modified edge 3: exposure=Perfluoroheptanoic acid, sc_gene=HMGCS2, relationship='ComptoxAI and Organoid association', weight=2
Modified edge 4: exposure=Perfluoroheptanoic acid, sc_gene=APOA2, relationship='ComptoxAI and Organoid association', weight=2
Added new edge 5: exposure=Perfluoroheptanoic acid, sc_gene=MT1F, relationship='Organoid association', weight=1
Modified edge 6: exposure=Perfluoroheptanoic acid, sc_gene=ACSL1, relationship='ComptoxAI and Organoid association', weight=2
Added new edge 7: exposu

## 2.e. Plot Graph

In [ ]:
# Color mapping
color_map = {
    'PFAS': 'Pink',
    'Disease': 'red',
    'Pathway': 'grey',
    'Gene: In-vitro': 'blue',
    'SNP': 'red',
    'Gene: eQTL': 'pink', 
    'Gene: SNP associated gene': 'pink',
    'Gene: from ComptoxAI': 'purple'
}

# Compute node colors based on 'type' attribute
node_colors = [color_map[G.nodes[node]['type']] for node in G]

# Draw the graph
plt.figure(figsize=(12, 8))  # Set the figure size
pos = nx.spring_layout(G)  # Layout for the nodes
nx.draw_networkx(G, pos, with_labels=True, node_color=node_colors, node_size=70, edge_color='grey', linewidths=1, font_size=10, arrows=True)
#nx.draw_networkx_edge_labels(G, pos, edge_labels=nx.get_edge_attributes(G, 'relationship'))

# plt.show()

## 2.f. Save Graph

In [ ]:
# Save graph
file_path = dir_res / 'ComptoxAI' / 'PFAS_prot_scRNAseq_HCC_082224.graphml'

# Write the graph to a GraphML file
nx.write_graphml(G, file_path)
print(f"Network saved to {file_path}") 

# 3. Diagnostics

In [ ]:
len(G.nodes)
temp_nodes = [node for node, data in G.nodes(data=True) if data.get('type') == 'Gene: SNP associated gene']

print(unique_gene_symbols_list)

## 3.a Identify disconnected graphs


In [ ]:
# Generate weakly connected components
weakly_connected_components = nx.weakly_connected_components(G)

# Get the size of each component (number of nodes)
component_sizes = [len(component) for component in weakly_connected_components]
# print(Counter(component_sizes))

# Identify genes not in the large componenet -----------

# Find all weakly connected components
weakly_connected_components = list(nx.weakly_connected_components(G))

# Find the largest weakly connected component (by number of nodes)
largest_component = max(weakly_connected_components, key=len)

# Find all nodes in the graph
all_nodes = set(G.nodes())

# Find nodes not in the largest component
nodes_not_in_largest = all_nodes - largest_component

# Prepare the data for the table
data = []
for node in nodes_not_in_largest:
    node_type = G.nodes[node].get('type', 'Unknown')  # Use 'Unknown' if no type is provided
    data.append((node, node_type))

# Create a DataFrame
df = pd.DataFrame(data, columns=['Node', 'Type'])

# Sort the DataFrame by the 'Type' column
df_sorted = df.sort_values(by='Type')

# Display the sorted DataFrame
#print(df_sorted)

# Remove these nodes from the graph
G_trim = G.copy()
G_trim.remove_nodes_from(nodes_not_in_largest)

print(len(G.nodes))
print(len(G_trim.nodes))
print(len(nodes_not_in_largest))
print(len(G_trim.nodes) == len(G.nodes)-len(nodes_not_in_largest))

# 3. Infomap

In [ ]:
# # Write pajek to test on https://www.mapequation.org/infomap/#Input
# Create a copy of the graph without node attributes
G_trim_no_attrs = nx.Graph()

# Add nodes and edges to the new graph
G_trim_no_attrs.add_nodes_from(G_trim.nodes())
G_trim_no_attrs.add_edges_from(G_trim.edges())

# Write the new graph to a Pajek file
nx.write_pajek(G_trim_no_attrs, "test_pajek_out.net")

In [ ]:
from infomap import Infomap

# Create a mapping from original node labels to integers
mapping = {node: i for i, node in enumerate(G_trim.nodes())}
reverse_mapping = {i: node for node, i in mapping.items()}

# Relabel the graph using the integer mapping
G_trim_int = nx.relabel_nodes(G_trim, mapping)

# Initialize 2 level Infomap
infomap = Infomap("--two-level --directed --flow-model directed -v --ftree")

# Initialize multi-level Infomap
# infomap = Infomap("--directed --flow-model directed -v ")

# Add graph to Infomap
infomap.add_networkx_graph(G_trim)

# Run Infomap algorithm
infomap.run()


In [ ]:
# Extract top-level map
network_flow = infomap.activeNetwork()

top_modules = infomap.modules()

# # # Create a new graph for the top-level map
# G_top_level = nx.DiGraph()

# # # Add nodes representing communities
# for module in top_modules:
#     G_top_level.add_node(module)

# # # Add weighted edges based on flow between communities
# for link in network_flow:
#     source_module = link[0]
#     target_module = link[1]
#     flow = link[2]
#     if source_module != target_module:  # Exclude self-loops if desired
#         if G_top_level.has_edge(source_module, target_module):
#             G_top_level[source_module][target_module]['weight'] += flow
#         else:
#             G_top_level.add_edge(source_module, target_module, weight=flow)